# 🎮 Rock Paper Scissors AI Game
## Computer Vision Hand Gesture Recognition

This notebook demonstrates a real-time Rock Paper Scissors game using:
- **MediaPipe** for hand tracking
- **OpenCV** for video processing
- **Hand gesture recognition** to detect Rock, Paper, or Scissors

---

## 📦 Step 1: Install Required Packages

Run this cell to install all dependencies:

In [ ]:
!pip install mediapipe==0.10.13
!pip install numpy==2.2.6
!pip install opencv-python==4.12.0.88
!pip install opencv-contrib-python==4.13.0.92

## 📚 Step 2: Import Libraries

Import all necessary libraries for the game:

In [ ]:
import cv2
import time
import random
import mediapipe as mp
import os
import numpy as np

print("✅ All libraries imported successfully!")
print(f"OpenCV Version: {cv2.__version__}")
print(f"MediaPipe Version: {mp.__version__}")

✅ All libraries imported successfully!
OpenCV Version: 4.12.0
MediaPipe Version: 0.10.13


## 🖼️ Step 3: Setup Emoji/Image Loading Function

This function loads and resizes emoji images for visual feedback:

In [ ]:
# Directory setup
BASE_DIR = os.getcwd()
EMOJI_DIR = os.path.join(BASE_DIR, "emojis")

# Create emojis directory if it doesn't exist
os.makedirs(EMOJI_DIR, exist_ok=True)

def load_emoji(filename, size=(120, 120)):
    """
    Load and resize an emoji image.
    
    Args:
        filename: Name of the emoji file
        size: Tuple of (width, height) for resizing
    
    Returns:
        Resized image or None if not found
    """
    path = os.path.join(EMOJI_DIR, filename)
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    
    if img is None:
        print(f"⚠️  Emoji not found at {path}")
        return None
    
    return cv2.resize(img, size)

# Try to load emojis (will show warnings if files don't exist)
EMOJI_IMG = {
    "Rock": load_emoji("rock.jpg"),
    "Paper": load_emoji("paper.png"),
    "Scissors": load_emoji("scissors.png"),
    None: None
}

print("\n📁 Emoji directory created at:", EMOJI_DIR)
print("\nℹ️  Note: Place your emoji images (rock.jpg, paper.png, scissors.png) in the 'emojis' folder")
print("   The game will work without them, but emojis enhance the visual experience!")

⚠️  Emoji not found at c:\Users\Prakhar\OneDrive\Desktop\webAsii\python\emojis\rock.jpg
⚠️  Emoji not found at c:\Users\Prakhar\OneDrive\Desktop\webAsii\python\emojis\paper.png
⚠️  Emoji not found at c:\Users\Prakhar\OneDrive\Desktop\webAsii\python\emojis\scissors.png

📁 Emoji directory created at: c:\Users\Prakhar\OneDrive\Desktop\webAsii\python\emojis

ℹ️  Note: Place your emoji images (rock.jpg, paper.png, scissors.png) in the 'emojis' folder
   The game will work without them, but emojis enhance the visual experience!


## 🎨 Step 4: Image Overlay Function

Function to overlay PNG images with transparency support:

In [ ]:
def overlay_png(bg, fg, x, y):
    """
    Overlay a foreground image on a background image at position (x, y).
    Supports PNG transparency (alpha channel).
    
    Args:
        bg: Background image
        fg: Foreground image (can have alpha channel)
        x: X position to place the foreground image
        y: Y position to place the foreground image
    
    Returns:
        Combined image
    """
    if fg is None:
        return bg
    
    h, w = fg.shape[:2]
    
    # Check if image has alpha channel (transparency)
    if fg.shape[2] == 4:
        alpha = fg[:, :, 3] / 255.0
        for c in range(3):
            bg[y:y+h, x:x+w, c] = (
                alpha * fg[:, :, c] +
                (1 - alpha) * bg[y:y+h, x:x+w, c]
            )
    else:
        # No alpha channel → simple overlay
        bg[y:y+h, x:x+w] = fg
    
    return bg

print("✅ Image overlay function defined!")

✅ Image overlay function defined!


## 🖐️ Step 5: Hand Gesture Recognition Functions

Functions to detect finger states and recognize gestures:

In [ ]:
def get_finger_states(hand_landmarks):
    """
    Detect which fingers are extended based on hand landmarks.
    
    Args:
        hand_landmarks: MediaPipe hand landmarks object
    
    Returns:
        Dictionary with finger states (True = extended, False = folded)
    """
    fingers = {}
    
    # Thumb (ignore for gesture recognition in this game)
    fingers["thumb"] = hand_landmarks.landmark[4].x > hand_landmarks.landmark[3].x
    
    # Other fingers: compare tip (8,12,16,20) with middle joint (6,10,14,18)
    # If tip is above middle joint (lower y value), finger is extended
    fingers["index"] = hand_landmarks.landmark[8].y < hand_landmarks.landmark[6].y
    fingers["middle"] = hand_landmarks.landmark[12].y < hand_landmarks.landmark[10].y
    fingers["ring"] = hand_landmarks.landmark[16].y < hand_landmarks.landmark[14].y
    fingers["pinky"] = hand_landmarks.landmark[20].y < hand_landmarks.landmark[18].y
    
    return fingers


def finger_to_move(fingers):
    """
    Convert finger states to Rock/Paper/Scissors move.
    
    Rules:
    - All fingers closed → Rock ✊
    - Index + Middle extended → Scissors ✌️
    - All fingers extended → Paper ✋
    
    Args:
        fingers: Dictionary of finger states
    
    Returns:
        "Rock", "Paper", "Scissors", or None
    """
    # Rock: all fingers closed
    if not fingers["index"] and not fingers["middle"] and not fingers["ring"] and not fingers["pinky"]:
        return "Rock"
    
    # Scissors: index and middle extended only
    if fingers["index"] and fingers["middle"] and not fingers["ring"] and not fingers["pinky"]:
        return "Scissors"
    
    # Paper: all fingers extended
    if fingers["index"] and fingers["middle"] and fingers["ring"] and fingers["pinky"]:
        return "Paper"
    
    return None

print("✅ Gesture recognition functions defined!")
print("\n🖐️  Gesture Guide:")
print("   Rock ✊     → All fingers closed")
print("   Scissors ✌️ → Index + Middle fingers extended")
print("   Paper ✋    → All fingers extended")

✅ Gesture recognition functions defined!

🖐️  Gesture Guide:
   Rock ✊     → All fingers closed
   Scissors ✌️ → Index + Middle fingers extended
   Paper ✋    → All fingers extended


## 🏆 Step 6: Game Logic Function

Function to determine the winner:

In [ ]:
def decide_winner(player, ai):
    """
    Determine the winner of Rock Paper Scissors.
    
    Rules:
    - Rock beats Scissors
    - Scissors beats Paper
    - Paper beats Rock
    
    Args:
        player: Player's move
        ai: AI's move
    
    Returns:
        "You", "AI", or "Draw"
    """
    if player == ai:
        return "Draw"
    
    if (
        (player == "Rock" and ai == "Scissors") or
        (player == "Paper" and ai == "Rock") or
        (player == "Scissors" and ai == "Paper")
    ):
        return "You"
    
    return "AI"

print("✅ Game logic function defined!")
print("\n🎮 Game Rules:")
print("   Rock ✊ beats Scissors ✌️")
print("   Scissors ✌️ beats Paper ✋")
print("   Paper ✋ beats Rock ✊")

✅ Game logic function defined!

🎮 Game Rules:
   Rock ✊ beats Scissors ✌️
   Scissors ✌️ beats Paper ✋
   Paper ✋ beats Rock ✊


## 🎥 Step 7: Initialize MediaPipe and Camera

Setup hand tracking and webcam:

In [ ]:
# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,              # Track only one hand
    min_detection_confidence=0.7,  # Minimum confidence for detection
    min_tracking_confidence=0.7    # Minimum confidence for tracking
)
mp_draw = mp.solutions.drawing_utils

print("✅ MediaPipe Hands initialized!")
print("   - Max hands: 1")
print("   - Detection confidence: 0.7")
print("   - Tracking confidence: 0.7")
print("\n⚠️  Note: Make sure your webcam is connected and accessible!")

✅ MediaPipe Hands initialized!
   - Max hands: 1
   - Detection confidence: 0.7
   - Tracking confidence: 0.7

⚠️  Note: Make sure your webcam is connected and accessible!


## 🚀 Step 8: Run the Game!

**Instructions:**
1. Make sure your webcam is connected
2. Position your hand in front of the camera
3. Make a gesture when the countdown reaches 0
4. Press **'q'** to quit the game

**Game Flow:**
- 3-second countdown
- Make your gesture (Rock/Paper/Scissors)
- See the result and score
- Game auto-resets after 8 seconds

In [ ]:
# Initialize webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Error: Could not open webcam!")
    print("   Please check your webcam connection and try again.")
else:
    print("✅ Webcam initialized successfully!")
    print("\n🎮 Starting the game...")
    print("\n📝 Controls:")
    print("   Press 'q' to quit")
    print("\n" + "="*50)
    
    # Game State Variables
    MOVES = ["Rock", "Paper", "Scissors"]
    ai_move = None
    human_move = None
    winner = None
    
    ai_score = 0
    human_score = 0
    
    start_time = time.time()
    moves_locked = False
    
    # Main Game Loop
    try:
        while True:
            success, frame = cap.read()
            if not success:
                print("⚠️  Failed to read frame from webcam")
                break
            
            # Flip frame for mirror effect
            frame = cv2.flip(frame, 1)
            h, w, _ = frame.shape
            
            # Convert to RGB for MediaPipe
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)
            
            detected_move = None
            
            # Process hand landmarks if detected
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    # Draw hand landmarks on frame
                    mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    
                    # Detect gesture
                    fingers = get_finger_states(hand_landmarks)
                    detected_move = finger_to_move(fingers)
            
            elapsed = time.time() - start_time
            
            # Phase 1: Countdown (0-3 seconds)
            if elapsed < 3:
                countdown = 3 - int(elapsed)
                cv2.putText(
                    frame, str(countdown),
                    (w // 2 - 30, h // 2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    4, (0, 0, 255), 6
                )
            
            # Phase 2: Lock Moves (at 3 seconds)
            elif not moves_locked:
                ai_move = random.choice(MOVES)
                human_move = detected_move
                winner = decide_winner(human_move, ai_move)
                
                # Update scores
                if winner == "You":
                    human_score += 1
                elif winner == "AI":
                    ai_score += 1
                
                moves_locked = True
            
            # Phase 3: Display Result (3-8 seconds)
            else:
                # Display AI move
                cv2.putText(frame, f"AI: {ai_move}", (50, 80),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 0, 0), 3)
                frame = overlay_png(frame, EMOJI_IMG[ai_move], 50, 120)
                
                # Display player move
                cv2.putText(frame, f"You: {human_move}", (w - 350, 80),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 0), 3)
                frame = overlay_png(frame, EMOJI_IMG[human_move], w - 200, 120)
                
                # Display winner
                cv2.putText(frame, f"Winner: {winner}",
                           (w // 2 - 200, h // 2),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 4)
                
                # Display score
                cv2.putText(frame, f"Score  You: {human_score}  AI: {ai_score}",
                           (w // 2 - 250, h - 40),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
                
                # Auto reset after 8 seconds
                if elapsed > 8:
                    start_time = time.time()
                    moves_locked = False
                    ai_move = None
                    human_move = None
                    winner = None
            
            # Show frame
            cv2.imshow("Rock Paper Scissors - AI", frame)
            
            # Check for quit key
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("\n🎮 Game ended by user")
                break
    
    except KeyboardInterrupt:
        print("\n🎮 Game interrupted by user")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        
        print("\n" + "="*50)
        print("📊 Final Score:")
        print(f"   You: {human_score}")
        print(f"   AI:  {ai_score}")
        
        if human_score > ai_score:
            print("\n🏆 Congratulations! You won!")
        elif ai_score > human_score:
            print("\n🤖 AI wins! Better luck next time!")
        else:
            print("\n🤝 It's a tie!")
        
        print("\n✅ Resources released successfully")

✅ Webcam initialized successfully!

🎮 Starting the game...

📝 Controls:
   Press 'q' to quit


🎮 Game ended by user

📊 Final Score:
   You: 5
   AI:  39

🤖 AI wins! Better luck next time!

✅ Resources released successfully


## 📝 Additional Notes

### Troubleshooting:

1. **Webcam not detected:**
   - Check if your webcam is connected
   - Try changing `cv2.VideoCapture(0)` to `cv2.VideoCapture(1)` if you have multiple cameras

2. **Hand not detected:**
   - Ensure good lighting
   - Keep your hand clearly visible in frame
   - Adjust `min_detection_confidence` value (lower = more sensitive)

3. **Gestures not recognized properly:**
   - Make clear, distinct gestures
   - Keep your hand steady during countdown
   - Ensure fingers are fully extended for Paper

### Customization Ideas:

- Add sound effects
- Create custom emoji images
- Adjust countdown duration
- Add more gesture recognition patterns
- Implement best-of-N rounds
- Add difficulty levels

---

**Created for Bootcamp Demo** | Python 3.9+ | OpenCV + MediaPipe